# 03 - Master Data Preparation

Notebook này chuẩn bị các bảng master/dimension để dùng cho mô hình dữ liệu và Power BI.

## Mục tiêu
- Load Product Master, Distribution Channel, Calendar, COGS, Retail Price, Classification
- Kiểm tra schema, missing values, duplicate, candidate keys
- Chuẩn hóa tên cột và kiểu dữ liệu
- Tạo `DimProduct`
- Tạo `DimStoreChannel`
- Tạo `DimDate`
- Chuẩn bị COGS / Retail Price cho bước enrich FactSales
- Kiểm tra khả năng join với Sales và Inventory
- Lưu các bảng dimension đã chuẩn hóa


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Khai báo đường dẫn

In [3]:
RAW_DATA_PATH = Path("../data/raw")
MASTER_PATH = RAW_DATA_PATH / "MasterData"
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Master path exists:", MASTER_PATH.exists())
print("Processed path exists:", PROCESSED_PATH.exists())

Master path exists: True
Processed path exists: True


## 2. Load các file master

In [4]:
product = pd.read_excel(MASTER_PATH / "Productmaster.xlsx")
distribution = pd.read_excel(MASTER_PATH / "Distribution Channel.xlsx")
calendar = pd.read_excel(MASTER_PATH / "Master_Calendar.xlsx", sheet_name="Calendar")
cogs = pd.read_excel(MASTER_PATH / "COGS.xlsx")
retail_price = pd.read_excel(MASTER_PATH / "Retail_price.xlsx")
classification = pd.read_excel(MASTER_PATH / "New_Core_Classification.xlsx")

print("Product:", product.shape)
print("Distribution:", distribution.shape)
print("Calendar:", calendar.shape)
print("COGS:", cogs.shape)
print("Retail Price:", retail_price.shape)
print("Classification:", classification.shape)

Product: (94867, 27)
Distribution: (3405, 24)
Calendar: (260, 8)
COGS: (522675, 5)
Retail Price: (669460, 6)
Classification: (150209, 11)


## 3. Xem nhanh schema từng bảng

In [5]:
master_tables = {
    "product": product,
    "distribution": distribution,
    "calendar": calendar,
    "cogs": cogs,
    "retail_price": retail_price,
    "classification": classification
}

for name, df in master_tables.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3))


PRODUCT
Shape: (94867, 27)
Columns: ['Unnamed: 0', 'index', 'color', 'color_group', 'listing_price', 'price_group', 'gender', 'product_group', 'detail_product_group', 'shoe_product', 'size_group', 'size', 'age_group', 'activity_group', 'image_copyright', 'lifestyle_group', 'launch_season', 'mold_code', 'heel_height', 'code_lock', 'option', 'cost_price', 'product_id', 'product_syle_color', 'product_syle', 'brand_name', 'vendor_name']
   Unnamed: 0  index color color_group  listing_price price_group gender  \
0           0      0   DEN         TỐI       255273.0     200<300    MEN   
1           1      1   DEN         TỐI       255273.0     200<300    MEN   
2           2      2   DEN         TỐI       255273.0     200<300    MEN   

  product_group detail_product_group shoe_product size_group  size  \
0           SAN                SANTD          STT   Ngoại lệ  38.0   
1           SAN                SANTD          STT   Ngoại lệ  39.0   
2           SAN                SANTD          S

## 4. Hàm tạo Data Quality Report

In [6]:
def quality_report(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "rows": len(df),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=False)
    })

In [7]:
product_quality = quality_report(product)
distribution_quality = quality_report(distribution)
calendar_quality = quality_report(calendar)
cogs_quality = quality_report(cogs)
retail_price_quality = quality_report(retail_price)
classification_quality = quality_report(classification)

product_quality

,dtype,rows,missing,missing_pct,unique
Unnamed: 0,int64,94867,0,0.00,94867
index,int64,94867,0,0.00,94867
color,str,94867,45,0.05,80
color_group,str,94867,0,0.00,7
listing_price,float64,94867,27,0.03,604
price_group,str,94867,27,0.03,8
gender,str,94867,0,0.00,4
product_group,str,94867,0,0.00,5
detail_product_group,str,94867,0,0.00,9
shoe_product,str,94867,0,0.00,29


## 5. Product Master - kiểm tra candidate key

In [8]:
print("Rows:", len(product))
print("Unique product_id:", product["product_id"].nunique(dropna=False))
print("Missing product_id:", product["product_id"].isna().sum())
print("Duplicate product_id rows:", product.duplicated(subset=["product_id"]).sum())

Rows: 94867
Unique product_id: 94867
Missing product_id: 0
Duplicate product_id rows: 0


In [9]:
product[
    product.duplicated(subset=["product_id"], keep=False)
].sort_values("product_id").head(20)

,Unnamed: 0,index,color,color_group,listing_price,price_group,gender,product_group,detail_product_group,shoe_product,size_group,size,age_group,activity_group,image_copyright,lifestyle_group,launch_season,mold_code,heel_height,code_lock,option,cost_price,product_id,product_syle_color,product_syle,brand_name,vendor_name


### Kiểm tra các cột kỹ thuật

`Unnamed: 0` và `index` nhiều khả năng là cột index được export từ hệ thống nguồn.

In [10]:
technical_cols_product = [
    col for col in ["Unnamed: 0", "index"]
    if col in product.columns
]

technical_cols_product

['Unnamed: 0', 'index']

## 6. Chuẩn hóa Product Master

In [11]:
dim_product = product.copy()

dim_product = dim_product.drop(
    columns=technical_cols_product,
    errors="ignore"
)

# Chuẩn hóa typo / tên cột nếu cần
rename_product = {
    "product_syle_color": "product_style_color",
    "product_syle": "product_style"
}

dim_product = dim_product.rename(columns=rename_product)

print(dim_product.shape)
dim_product.head()

(94867, 25)


,color,color_group,listing_price,price_group,gender,product_group,detail_product_group,shoe_product,size_group,size,age_group,activity_group,image_copyright,lifestyle_group,launch_season,mold_code,heel_height,code_lock,option,cost_price,product_id,product_style_color,product_style,brand_name,vendor_name
0,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,38.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,80e1107e5bf74598baffea3a7b6073c5DEN38,80e1107e5bf74598baffea3a7b6073c5DEN,80e1107e5bf74598baffea3a7b6073c5,Brand1,vendor0
1,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,39.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,c8223e6133a64491a006dc0f95c2bfd9DEN39,c8223e6133a64491a006dc0f95c2bfd9DEN,c8223e6133a64491a006dc0f95c2bfd9,Brand1,vendor0
2,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,40.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,bec30e131ee04e49a4c87bc56f135b13DEN40,bec30e131ee04e49a4c87bc56f135b13DEN,bec30e131ee04e49a4c87bc56f135b13,Brand1,vendor0
3,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,41.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,3f4e265b0ac740e9b9edfd23e0ba1ca5DEN41,3f4e265b0ac740e9b9edfd23e0ba1ca5DEN,3f4e265b0ac740e9b9edfd23e0ba1ca5,Brand1,vendor0
4,DEN,TỐI,255273.0,200<300,MEN,SAN,SANTD,STT,Ngoại lệ,42.0,16 đến <24 tuổi,Thường nhật/Trường học,CƠ BẢN,Sport,07DE,DRB057,Đế Sẹp,x,P03-C05,176800.0,53e6284597944ec787e002b631391210DEN42,53e6284597944ec787e002b631391210DEN,53e6284597944ec787e002b631391210,Brand1,vendor0


### Kiểm tra uniqueness sau khi bỏ technical columns

In [12]:
product_key_check = pd.Series({
    "rows": len(dim_product),
    "unique_product_id": dim_product["product_id"].nunique(dropna=False),
    "missing_product_id": dim_product["product_id"].isna().sum(),
    "duplicate_product_id_rows": dim_product.duplicated(subset=["product_id"]).sum()
})

product_key_check

rows                         94867
unique_product_id            94867
missing_product_id               0
duplicate_product_id_rows        0
dtype: int64

> Nếu `product_id` unique và không missing, có thể dùng làm natural key cho `DimProduct`.

## 7. Phân tích missing quan trọng trong Product

In [13]:
important_product_cols = [
    "product_id",
    "brand_name",
    "product_group",
    "detail_product_group",
    "gender",
    "size",
    "color",
    "listing_price",
    "cost_price",
    "vendor_name"
]

existing_product_cols = [
    c for c in important_product_cols
    if c in dim_product.columns
]

dim_product[existing_product_cols].isna().sum().sort_values(ascending=False)

vendor_name             5423
cost_price               235
size                     145
color                     45
listing_price             27
product_id                 0
brand_name                 0
gender                     0
product_group              0
detail_product_group       0
dtype: int64

Không fill các field như `mold_code`, `code_lock`, `heel_height` một cách máy móc vì nhiều null có thể là hợp lệ theo loại sản phẩm.

## 8. Distribution Channel - kiểm tra candidate keys

In [14]:
print("Rows:", len(distribution))

candidate_cols = [
    "site_store",
    "channel_id",
    "customer_id",
    "customer_name"
]

for col in candidate_cols:
    if col in distribution.columns:
        print(
            f"{col}: unique={distribution[col].nunique(dropna=False)}, "
            f"missing={distribution[col].isna().sum()}"
        )

Rows: 3405
site_store: unique=3158, missing=0
channel_id: unique=4, missing=0
customer_id: unique=3405, missing=0
customer_name: unique=2037, missing=0


In [15]:
print(
    "Duplicate site_store rows:",
    distribution.duplicated(subset=["site_store"]).sum()
)

print(
    "Duplicate customer_id rows:",
    distribution.duplicated(subset=["customer_id"]).sum()
)

print(
    "Duplicate site_store + customer_id rows:",
    distribution.duplicated(
        subset=["site_store", "customer_id"]
    ).sum()
)

Duplicate site_store rows: 247
Duplicate customer_id rows: 0
Duplicate site_store + customer_id rows: 0


### Kiểm tra mapping site_store ↔ customer_id

In [16]:
distribution[
    ["site_store", "customer_id", "channel_id", "region", "city_level"]
].sort_values(["site_store", "customer_id"]).head(30)

,site_store,customer_id,channel_id,region,city_level
3349,1100,0c7e0c90b,TGPP,KVMN,Cấp TW
3348,1100,5126f889d,TGPP,KVMN,Còn lại
1004,1100,70ea4e990,TGPP,KVMN,Còn lại
686,1101,dd5a95cfa,CHTT,KVMN,Cấp TW
687,1101,f56cfa95a,CHTT,KVMN,Cấp TW
688,1102,590a310a8,CHTT,KVMN,Cấp TW
689,1103,8a6174afa,CHTT,KVMN,Cấp TW
690,1103,8edb4cbc0,CHTT,KVMN,Cấp TW
691,1104,421de55a1,CHTT,KVMN,Cấp TW
3295,1104,eed77e1d6,CHTT,KVMN,Cấp TW


## 9. Chuẩn hóa Distribution Channel

In [17]:
dim_store_channel = distribution.copy()

technical_cols_distribution = [
    col for col in ["Unnamed: 0", "index"]
    if col in dim_store_channel.columns
]

dim_store_channel = dim_store_channel.drop(
    columns=technical_cols_distribution,
    errors="ignore"
)

dim_store_channel.head()

,site_store,b2b_b2c,channel_id,region,city_level,store_concept,trade_term,area_range,store_type,urbanization,branch_area,address_2,address_3,showroom_area,warehouse_area,start_month,start_year,end_month,end_year,note,customer_id,customer_name
0,60000003,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Bình Tân,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,435043cf9,customer878
1,60000006,B2B,ST,KVMN,Cấp 2,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Tp. Biên Hoà,ĐNI,NaN,NaN,8,2010,NaN,NaN,NaN,db3c83bfa,customer904
2,60000007,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. 10,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,f9bd8e994,customer1213
3,60000008,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Gò Vấp,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,c6fb2b695,customer1792
4,60000534,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Phú Nhuận,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,68be208f8,customer463


### Tạo surrogate key

Do `site_store` hoặc `customer_id` có thể không unique riêng lẻ, ta tạo `store_channel_key` làm key kỹ thuật cho dimension.

In [18]:
dim_store_channel = dim_store_channel.reset_index(drop=True)

dim_store_channel["store_channel_key"] = (
    dim_store_channel.index + 1
)

cols = ["store_channel_key"] + [
    c for c in dim_store_channel.columns
    if c != "store_channel_key"
]

dim_store_channel = dim_store_channel[cols]

dim_store_channel.head()

,store_channel_key,site_store,b2b_b2c,channel_id,region,city_level,store_concept,trade_term,area_range,store_type,urbanization,branch_area,address_2,address_3,showroom_area,warehouse_area,start_month,start_year,end_month,end_year,note,customer_id,customer_name
0,1,60000003,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Bình Tân,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,435043cf9,customer878
1,2,60000006,B2B,ST,KVMN,Cấp 2,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Tp. Biên Hoà,ĐNI,NaN,NaN,8,2010,NaN,NaN,NaN,db3c83bfa,customer904
2,3,60000007,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. 10,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,f9bd8e994,customer1213
3,4,60000008,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Gò Vấp,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,c6fb2b695,customer1792
4,5,60000534,B2B,ST,KVMN,Cấp TW,Cửa hàng của các siêu thị,Mua đứt bán đoạn,NaN,Siêu thị,Nội thành,CNMN,Q. Phú Nhuận,HCM,NaN,NaN,8,2010,NaN,NaN,NaN,68be208f8,customer463


## 10. Calendar - chuẩn hóa DimDate

In [19]:
calendar.head()

,Year,Month,Month_3Char,Week,YearWeek,Start Date,End Date,CNY
0,2016,1,1,1,201601,2016-01-04,2016-01-10,No
1,2016,1,1,2,201602,2016-01-11,2016-01-17,No
2,2016,1,1,3,201603,2016-01-18,2016-01-24,Yes
3,2016,1,1,4,201604,2016-01-25,2016-01-31,Yes
4,2016,2,2,5,201605,2016-02-01,2016-02-07,Yes


In [20]:
dim_date = calendar.copy()

dim_date = dim_date.rename(columns={
    "Year": "year",
    "Month": "month_number",
    "Month_3Char": "month_name_short",
    "Week": "week_number",
    "YearWeek": "year_week",
    "Start Date": "week_start_date",
    "End Date": "week_end_date",
    "CNY": "cny_flag"
})

dim_date["week_start_date"] = pd.to_datetime(
    dim_date["week_start_date"],
    errors="coerce"
)

dim_date["week_end_date"] = pd.to_datetime(
    dim_date["week_end_date"],
    errors="coerce"
)

dim_date.head()

,year,month_number,month_name_short,week_number,year_week,week_start_date,week_end_date,cny_flag
0,2016,1,1,1,201601,2016-01-04,2016-01-10,No
1,2016,1,1,2,201602,2016-01-11,2016-01-17,No
2,2016,1,1,3,201603,2016-01-18,2016-01-24,Yes
3,2016,1,1,4,201604,2016-01-25,2016-01-31,Yes
4,2016,2,2,5,201605,2016-02-01,2016-02-07,Yes


### Kiểm tra key `year_week`

In [21]:
date_key_check = pd.Series({
    "rows": len(dim_date),
    "unique_year_week": dim_date["year_week"].nunique(dropna=False),
    "missing_year_week": dim_date["year_week"].isna().sum(),
    "duplicate_year_week_rows": dim_date.duplicated(subset=["year_week"]).sum()
})

date_key_check

rows                        260
unique_year_week            260
missing_year_week             0
duplicate_year_week_rows      0
dtype: int64

## 11. COGS - chuẩn hóa

In [22]:
cogs_clean = cogs.copy()

cogs_clean = cogs_clean.drop(
    columns=[c for c in ["Unnamed: 0", "index"] if c in cogs_clean.columns],
    errors="ignore"
)

cogs_clean["valid_from"] = pd.to_datetime(
    cogs_clean["valid_from"],
    errors="coerce"
)

cogs_clean["valid_to"] = pd.to_datetime(
    cogs_clean["valid_to"],
    errors="coerce"
)

cogs_clean.head()

/var/folders/j3/rfsd83t914q9r5c7nvkkl_kh0000gn/T/ipykernel_32791/1198495338.py:8: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cogs_clean["valid_from"] = pd.to_datetime(
/var/folders/j3/rfsd83t914q9r5c7nvkkl_kh0000gn/T/ipykernel_32791/1198495338.py:13: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cogs_clean["valid_to"] = pd.to_datetime(


,amount,valid_from,valid_to,product_id
0,50000,2021-12-23,2023-01-31,904de55fb0ca414cb86b2e466c9d74c1OOO00
1,50000,2021-12-23,2023-01-31,904de55fb0ca414cb86b2e466c9d74c1OOO00
2,479400,2023-01-01,2023-02-07,9e8ecc5d2b104d609e3b53c4724bc96aCAM39
3,479400,2023-01-01,2023-02-07,9225ed5b408343be98a4dc63d4027a26CAM40
4,479400,2023-01-01,2023-02-07,d1410b40bb004d1f9b2c2619d2c933f6CAM41


In [23]:
print("Rows:", len(cogs_clean))
print("Unique product_id:", cogs_clean["product_id"].nunique(dropna=False))
print("Missing product_id:", cogs_clean["product_id"].isna().sum())
print(
    "Duplicate product_id rows:",
    cogs_clean.duplicated(subset=["product_id"]).sum()
)

Rows: 522675
Unique product_id: 209621
Missing product_id: 0
Duplicate product_id rows: 313054


> COGS có `valid_from` và `valid_to`, nên rất có thể đây là bảng giá vốn theo thời gian. Không được reduce còn 1 dòng/product nếu chưa chọn rule theo hiệu lực ngày.

## 12. Retail Price - chuẩn hóa

In [24]:
retail_price_clean = retail_price.copy()

retail_price_clean = retail_price_clean.drop(
    columns=[
        c for c in ["Unnamed: 0", "index"]
        if c in retail_price_clean.columns
    ],
    errors="ignore"
)

retail_price_clean["valid_from"] = pd.to_datetime(
    retail_price_clean["valid_from"],
    errors="coerce"
)

retail_price_clean["valid_to"] = pd.to_datetime(
    retail_price_clean["valid_to"],
    errors="coerce"
)

retail_price_clean.head()

,amount,valid_from,valid_to,product_id
0,125000,2024-01-01,9999-01-12,f64e53d83cfa461abe156559d55ccae2DEN35
1,125000,2024-01-01,9999-01-12,c50a6b0caa0b4ca9af55cf27607b64d3DEN36
2,125000,2024-01-01,9999-01-12,408592eb75e44216a1f65e831f9a08e5DEN37
3,125000,2024-01-01,9999-01-12,9e7e6f15b9764540bebaca2de63abb03DEN38
4,125000,2024-01-01,9999-01-12,473397f29c1a4005b3e9bf75496f496eDEN39


In [25]:
print("Rows:", len(retail_price_clean))
print(
    "Unique product_id:",
    retail_price_clean["product_id"].nunique(dropna=False)
)
print(
    "Duplicate product_id rows:",
    retail_price_clean.duplicated(subset=["product_id"]).sum()
)

Rows: 669460
Unique product_id: 212409
Duplicate product_id rows: 457051


## 13. Product Classification - chuẩn hóa

In [26]:
classification_clean = classification.copy()

classification_clean = classification_clean.drop(
    columns=[
        c for c in ["Unnamed: 0", "index"]
        if c in classification_clean.columns
    ],
    errors="ignore"
)

classification_clean = classification_clean.rename(columns={
    "lauch_season": "launch_season",
    "lauch_season_num": "launch_season_num",
    "product_syle_color": "product_style_color"
})

classification_clean.head()

,launch_season,launch_season_num,sales_season,sales_season_num,final_status,b2c_assortment,b2b_assortment,total_assortment,product_style_color
0,NaN,0.0,17DE,1712.0,CORE,1.0,1.0,1.0,0e911e94ee4f4fbebcc4dca85e6496ebDOD
1,NaN,0.0,18AP,1804.0,CORE,1.0,1.0,1.0,0e911e94ee4f4fbebcc4dca85e6496ebDOD
2,NaN,0.0,18AU,1808.0,CORE,1.0,1.0,1.0,0e911e94ee4f4fbebcc4dca85e6496ebDOD
3,NaN,0.0,18DE,1812.0,CORE,1.0,1.0,1.0,0e911e94ee4f4fbebcc4dca85e6496ebDOD
4,NaN,0.0,19AP,1904.0,CORE,1.0,1.0,1.0,0e911e94ee4f4fbebcc4dca85e6496ebDOD


### Kiểm tra key `product_style_color`

In [27]:
if "product_style_color" in classification_clean.columns:
    classification_key_check = pd.Series({
        "rows": len(classification_clean),
        "unique_product_style_color":
            classification_clean["product_style_color"].nunique(dropna=False),
        "missing_product_style_color":
            classification_clean["product_style_color"].isna().sum(),
        "duplicate_product_style_color_rows":
            classification_clean.duplicated(
                subset=["product_style_color"]
            ).sum()
    })

    display(classification_key_check)

rows                                  150209
unique_product_style_color             12192
missing_product_style_color               33
duplicate_product_style_color_rows    138017
dtype: int64

## 14. Join Product với Classification

Join bằng `product_style_color` nếu cả hai bảng có key tương ứng.

In [28]:
if (
    "product_style_color" in dim_product.columns
    and "product_style_color" in classification_clean.columns
):
    dim_product_enriched = dim_product.merge(
        classification_clean,
        on="product_style_color",
        how="left",
        suffixes=("", "_class")
    )

    print("Before join:", dim_product.shape)
    print("After join:", dim_product_enriched.shape)

else:
    dim_product_enriched = dim_product.copy()
    print("Không tìm thấy product_style_color ở cả hai bảng.")

Before join: (94867, 25)
After join: (231368, 33)


### Kiểm tra row explosion sau join

In [29]:
print("Original product rows:", len(dim_product))
print("Enriched product rows:", len(dim_product_enriched))
print("Difference:", len(dim_product_enriched) - len(dim_product))

Original product rows: 94867
Enriched product rows: 231368
Difference: 136501


Nếu số dòng tăng sau join, nghĩa là classification key không unique và cần xử lý trước khi dùng làm dimension.

## 15. Load Sales Clean để kiểm tra foreign key coverage

In [30]:
sales_clean_path = PROCESSED_PATH / "sales_clean.pkl"
sales_raw_path = PROCESSED_PATH / "sales_combined_raw.pkl"

if sales_clean_path.exists():
    sales_check = pd.read_pickle(sales_clean_path)
    print("Loaded sales_clean.pkl")
elif sales_raw_path.exists():
    sales_check = pd.read_pickle(sales_raw_path)
    print("Loaded sales_combined_raw.pkl")
else:
    sales_check = None
    print("Không tìm thấy Sales pickle để kiểm tra FK coverage.")

Loaded sales_clean.pkl


## 16. Kiểm tra Sales → Product coverage

In [31]:
if sales_check is not None:
    sales_products = set(
        sales_check["product_id"].dropna().astype(str)
    )

    master_products = set(
        dim_product["product_id"].dropna().astype(str)
    )

    missing_products_in_master = (
        sales_products - master_products
    )

    print("Unique products in Sales:", len(sales_products))
    print("Unique products in Product Master:", len(master_products))
    print(
        "Sales products missing from Product Master:",
        len(missing_products_in_master)
    )

    print(
        "Coverage:",
        round(
            (1 - len(missing_products_in_master) / len(sales_products)) * 100,
            2
        ),
        "%"
    )

Unique products in Sales: 30367
Unique products in Product Master: 94867
Sales products missing from Product Master: 1
Coverage: 100.0 %


## 17. Kiểm tra Sales `site` với Distribution `site_store`

In [32]:
if sales_check is not None:
    sales_sites = set(
        sales_check["site"].dropna().astype(str)
    )

    distribution_sites = set(
        dim_store_channel["site_store"].dropna().astype(str)
    )

    missing_sites = sales_sites - distribution_sites

    print("Unique Sales sites:", len(sales_sites))
    print(
        "Unique Distribution site_store:",
        len(distribution_sites)
    )
    print(
        "Sales sites missing from Distribution:",
        len(missing_sites)
    )

    print(
        sorted(list(missing_sites))[:20]
    )

Unique Sales sites: 227
Unique Distribution site_store: 3158
Sales sites missing from Distribution: 0
[]


## 18. Kiểm tra Sales `week` với Calendar `year_week`

In [33]:
if sales_check is not None:
    sales_weeks = set(
        sales_check["week"].dropna().astype(str)
    )

    calendar_weeks = set(
        dim_date["year_week"].dropna().astype(str)
    )

    missing_weeks = sales_weeks - calendar_weeks

    print("Unique Sales weeks:", len(sales_weeks))
    print("Unique Calendar weeks:", len(calendar_weeks))
    print(
        "Sales weeks missing from Calendar:",
        len(missing_weeks)
    )

    print(
        sorted(list(missing_weeks))[:20]
    )

Unique Sales weeks: 85
Unique Calendar weeks: 260
Sales weeks missing from Calendar: 85
['202153', '202201', '202202', '202203', '202204', '202205', '202206', '202207', '202208', '202209', '202210', '202211', '202212', '202213', '202214', '202215', '202216', '202217', '202218', '202219']


## 19. Kiểm tra Inventory → Product coverage

Đọc toàn bộ Inventory snapshot và chỉ lấy `product_id`, `plant` để kiểm tra coverage.

In [34]:
INVENTORY_PATH = RAW_DATA_PATH / "Inventory_snapshot_data"

inventory_files = sorted(
    INVENTORY_PATH.glob("*.xlsx")
)

print("Inventory files:", len(inventory_files))

Inventory files: 12


In [35]:
inventory_key_parts = []

for file in inventory_files:
    temp = pd.read_excel(
        file,
        usecols=lambda c: c in [
            "product_id",
            "plant"
        ]
    )
    inventory_key_parts.append(temp)

inventory_keys = pd.concat(
    inventory_key_parts,
    ignore_index=True
)

print(inventory_keys.shape)
inventory_keys.head()

(1367080, 2)


,plant,product_id
0,1201,1259098aaa8e447181f13903f84e5db1OOO35
1,1201,39b38616e4d649ab9c3b7d04e82e079fOOO36
2,1201,8db14e88898e40a392f80ed69c30e206OOO37
3,1201,7c15de90afd343338f93c8f65a0d8380OOO38
4,1201,6c04a173aec34690b242ed4e09367e96OOO39


In [36]:
inventory_products = set(
    inventory_keys["product_id"]
    .dropna()
    .astype(str)
)

master_products = set(
    dim_product["product_id"]
    .dropna()
    .astype(str)
)

missing_inventory_products = (
    inventory_products - master_products
)

print(
    "Unique Inventory products:",
    len(inventory_products)
)

print(
    "Inventory products missing from Product Master:",
    len(missing_inventory_products)
)

if len(inventory_products) > 0:
    print(
        "Inventory Product coverage:",
        round(
            (
                1
                - len(missing_inventory_products)
                / len(inventory_products)
            ) * 100,
            2
        ),
        "%"
    )

Unique Inventory products: 26280
Inventory products missing from Product Master: 3
Inventory Product coverage: 99.99 %


## 20. Master Data Validation Summary

In [37]:
master_validation = pd.Series({
    "dim_product_rows": len(dim_product),
    "dim_product_unique_product_id":
        dim_product["product_id"].nunique(dropna=False),
    "dim_product_duplicate_product_id":
        dim_product.duplicated(subset=["product_id"]).sum(),

    "dim_store_channel_rows":
        len(dim_store_channel),

    "dim_date_rows":
        len(dim_date),
    "dim_date_duplicate_year_week":
        dim_date.duplicated(subset=["year_week"]).sum(),

    "cogs_rows":
        len(cogs_clean),

    "retail_price_rows":
        len(retail_price_clean),

    "classification_rows":
        len(classification_clean)
})

master_validation

dim_product_rows                     94867
dim_product_unique_product_id        94867
dim_product_duplicate_product_id         0
dim_store_channel_rows                3405
dim_date_rows                          260
dim_date_duplicate_year_week             0
cogs_rows                           522675
retail_price_rows                   669460
classification_rows                 150209
dtype: int64

## 21. Lưu các bảng đã chuẩn hóa

Dùng Pickle để tránh phụ thuộc `pyarrow` trên macOS hiện tại.

In [38]:
dim_product_path = PROCESSED_PATH / "dim_product.pkl"
dim_store_channel_path = PROCESSED_PATH / "dim_store_channel.pkl"
dim_date_path = PROCESSED_PATH / "dim_date.pkl"
cogs_path = PROCESSED_PATH / "cogs_clean.pkl"
retail_price_path = PROCESSED_PATH / "retail_price_clean.pkl"
classification_path = PROCESSED_PATH / "classification_clean.pkl"

dim_product.to_pickle(dim_product_path)
dim_store_channel.to_pickle(dim_store_channel_path)
dim_date.to_pickle(dim_date_path)
cogs_clean.to_pickle(cogs_path)
retail_price_clean.to_pickle(retail_price_path)
classification_clean.to_pickle(classification_path)

print("Saved:")
print(dim_product_path)
print(dim_store_channel_path)
print(dim_date_path)
print(cogs_path)
print(retail_price_path)
print(classification_path)

Saved:
../data/processed/dim_product.pkl
../data/processed/dim_store_channel.pkl
../data/processed/dim_date.pkl
../data/processed/cogs_clean.pkl
../data/processed/retail_price_clean.pkl
../data/processed/classification_clean.pkl


## 22. Kiểm tra file output

In [39]:
output_files = [
    dim_product_path,
    dim_store_channel_path,
    dim_date_path,
    cogs_path,
    retail_price_path,
    classification_path
]

for path in output_files:
    print(
        path.name,
        "exists =", path.exists(),
        "| size MB =",
        round(path.stat().st_size / 1024 / 1024, 2)
        if path.exists()
        else None
    )

dim_product.pkl exists = True | size MB = 19.86
dim_store_channel.pkl exists = True | size MB = 0.4
dim_date.pkl exists = True | size MB = 0.02
cogs_clean.pkl exists = True | size MB = 21.45
retail_price_clean.pkl exists = True | size MB = 25.61
classification_clean.pkl exists = True | size MB = 8.02


# 23. Bốn kiểm tra bổ sung trước khi thiết kế Star Schema

Phần này bổ sung đúng 4 vấn đề còn thiếu:

1. Sales ↔ Customer / Distribution
2. Calendar datatype / join key
3. Classification grain
4. COGS / Retail Price date validity

Mục tiêu của phần này là **kiểm tra relationship và grain**, chưa tự động xóa hay ép dữ liệu theo giả định.

## 23.1 Sales ↔ Customer / Distribution

Cần trả lời ba câu hỏi:

- `Distribution.customer_id` có unique không?
- Sales có match Distribution tốt hơn bằng `customer_id` hay bằng `(site, customer_id)`?
- Nếu join bằng `customer_id`, thông tin site/channel có nhất quán không?

In [40]:
def normalize_key(series):
    """Chuẩn hóa key để so sánh mà không làm thay đổi dữ liệu gốc."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

if sales_check is None:
    print("Không có sales_check. Hãy bảo đảm sales_clean.pkl hoặc sales_combined_raw.pkl tồn tại.")
else:
    sales_rel = sales_check.copy()
    dist_rel = dim_store_channel.copy()

    sales_rel["customer_key"] = normalize_key(sales_rel["customer_id"])
    sales_rel["site_key"] = normalize_key(sales_rel["site"])

    dist_rel["customer_key"] = normalize_key(dist_rel["customer_id"])
    dist_rel["site_key"] = normalize_key(dist_rel["site_store"])

    print("Distribution rows:", len(dist_rel))
    print("Unique customer_key:", dist_rel["customer_key"].nunique(dropna=False))
    print(
        "Duplicate customer_key rows:",
        dist_rel.duplicated(subset=["customer_key"]).sum()
    )
    print(
        "Duplicate (site_key, customer_key) rows:",
        dist_rel.duplicated(subset=["site_key", "customer_key"]).sum()
    )

Distribution rows: 3405
Unique customer_key: 3405
Duplicate customer_key rows: 0
Duplicate (site_key, customer_key) rows: 0


In [41]:
if sales_check is not None:
    sales_customer_keys = set(
        sales_rel["customer_key"].dropna()
    )
    dist_customer_keys = set(
        dist_rel["customer_key"].dropna()
    )

    missing_customer_keys = sales_customer_keys - dist_customer_keys

    print("Unique customer_id in Sales:", len(sales_customer_keys))
    print("Unique customer_id in Distribution:", len(dist_customer_keys))
    print("Sales customer_id missing from Distribution:", len(missing_customer_keys))

    if len(sales_customer_keys) > 0:
        customer_coverage = (
            1 - len(missing_customer_keys) / len(sales_customer_keys)
        ) * 100
        print("Customer coverage:", round(customer_coverage, 4), "%")

    print("\nSample customer_id không match:")
    print(sorted(list(missing_customer_keys))[:20])

Unique customer_id in Sales: 1174
Unique customer_id in Distribution: 3405
Sales customer_id missing from Distribution: 1
Customer coverage: 99.9148 %

Sample customer_id không match:
['UNKNOWN']


In [42]:
if sales_check is not None:
    sales_pairs = set(
        sales_rel[
            ["site_key", "customer_key"]
        ]
        .dropna()
        .itertuples(index=False, name=None)
    )

    dist_pairs = set(
        dist_rel[
            ["site_key", "customer_key"]
        ]
        .dropna()
        .itertuples(index=False, name=None)
    )

    missing_pairs = sales_pairs - dist_pairs

    print("Unique (site, customer) pairs in Sales:", len(sales_pairs))
    print("Unique (site_store, customer) pairs in Distribution:", len(dist_pairs))
    print("Sales pairs missing from Distribution:", len(missing_pairs))

    if len(sales_pairs) > 0:
        pair_coverage = (
            1 - len(missing_pairs) / len(sales_pairs)
        ) * 100
        print("Pair coverage:", round(pair_coverage, 4), "%")

    print("\nSample pair không match:")
    print(sorted(list(missing_pairs))[:20])

Unique (site, customer) pairs in Sales: 1418
Unique (site_store, customer) pairs in Distribution: 3405
Sales pairs missing from Distribution: 1079
Pair coverage: 23.9069 %

Sample pair không match:
[('1100', '007e6bb72'), ('1100', '008e9d4fd'), ('1100', '02003980e'), ('1100', '021e839cd'), ('1100', '02f0892fa'), ('1100', '03fcec181'), ('1100', '047ce954c'), ('1100', '04946638c'), ('1100', '04cdd95b1'), ('1100', '056bd098f'), ('1100', '05b512e2d'), ('1100', '05d403973'), ('1100', '05e94823a'), ('1100', '06456635f'), ('1100', '06a1558e6'), ('1100', '071cfd357'), ('1100', '0750a0818'), ('1100', '08dad4c83'), ('1100', '096090159'), ('1100', '0a91adadb')]


In [43]:
if sales_check is not None:
    # Vì customer_id đang unique trong Distribution, thử join theo customer_id
    # để kiểm tra site/channel có khớp với Sales không.
    dist_customer_lookup = (
        dist_rel[
            [
                "customer_key",
                "site_key",
                "channel_id"
            ]
        ]
        .drop_duplicates(subset=["customer_key"])
        .rename(columns={
            "site_key": "master_site_key",
            "channel_id": "master_channel_id"
        })
    )

    sales_dist_check = sales_rel.merge(
        dist_customer_lookup,
        on="customer_key",
        how="left"
    )

    sales_dist_check["site_match"] = (
        sales_dist_check["site_key"]
        == sales_dist_check["master_site_key"]
    )

    sales_dist_check["channel_match"] = (
        normalize_key(sales_dist_check["channel_id"])
        == normalize_key(sales_dist_check["master_channel_id"])
    )

    relationship_summary = pd.Series({
        "sales_rows": len(sales_dist_check),
        "rows_with_customer_match":
            sales_dist_check["master_site_key"].notna().sum(),
        "rows_site_match":
            sales_dist_check["site_match"].sum(),
        "rows_site_mismatch":
            (
                sales_dist_check["master_site_key"].notna()
                & ~sales_dist_check["site_match"]
            ).sum(),
        "rows_channel_match":
            sales_dist_check["channel_match"].sum(),
        "rows_channel_mismatch":
            (
                sales_dist_check["master_channel_id"].notna()
                & ~sales_dist_check["channel_match"]
            ).sum()
    })

    relationship_summary

In [44]:
if sales_check is not None:
    sales_dist_check[
        sales_dist_check["master_site_key"].notna()
        & ~sales_dist_check["site_match"]
    ][
        [
            "customer_id",
            "site",
            "master_site_key",
            "channel_id",
            "master_channel_id",
            "product_id",
            "week"
        ]
    ].head(20)

## 23.2 Calendar datatype / join key



In [45]:
if sales_check is not None:
    print("Sales week dtype:", sales_check["week"].dtype)
    print("Calendar year_week dtype:", dim_date["year_week"].dtype)

    print("\nSales week samples:")
    print(sales_check["week"].drop_duplicates().head(10).tolist())

    print("\nCalendar year_week samples:")
    print(dim_date["year_week"].drop_duplicates().head(10).tolist())

Sales week dtype: int64
Calendar year_week dtype: int64

Sales week samples:
[202201, 202204, 202202, 202203, 202153, 202205, 202301, 202302, 202305, 202303]

Calendar year_week samples:
[201601, 201602, 201603, 201604, 201605, 201606, 201607, 201608, 201609, 201610]


In [46]:
def normalize_integer_like_key(series):
    """
    Chuẩn hóa các key kiểu 202201, 202201.0, '202201 ' thành '202201'.
    Không ghi đè cột gốc.
    """
    s = series.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    return s

if sales_check is not None:
    sales_week_key = normalize_integer_like_key(sales_check["week"])
    calendar_week_key = normalize_integer_like_key(dim_date["year_week"])

    sales_week_set = set(sales_week_key.dropna())
    calendar_week_set = set(calendar_week_key.dropna())

    missing_after_normalization = (
        sales_week_set - calendar_week_set
    )

    print("Unique Sales weeks:", len(sales_week_set))
    print("Unique Calendar weeks:", len(calendar_week_set))
    print(
        "Sales weeks missing AFTER normalization:",
        len(missing_after_normalization)
    )

    if len(sales_week_set) > 0:
        week_coverage = (
            1 - len(missing_after_normalization) / len(sales_week_set)
        ) * 100
        print("Week coverage:", round(week_coverage, 4), "%")

    print("\nMissing weeks:")
    print(sorted(list(missing_after_normalization)))

Unique Sales weeks: 85
Unique Calendar weeks: 260
Sales weeks missing AFTER normalization: 85
Week coverage: 0.0 %

Missing weeks:
['202153', '202201', '202202', '202203', '202204', '202205', '202206', '202207', '202208', '202209', '202210', '202211', '202212', '202213', '202214', '202215', '202216', '202217', '202218', '202219', '202220', '202221', '202222', '202223', '202224', '202225', '202226', '202227', '202228', '202229', '202230', '202231', '202232', '202233', '202234', '202235', '202236', '202237', '202238', '202239', '202240', '202241', '202242', '202243', '202244', '202245', '202246', '202247', '202248', '202249', '202250', '202251', '202252', '202301', '202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310', '202311', '202312', '202313', '202314', '202315', '202316', '202317', '202318', '202319', '202320', '202321', '202322', '202323', '202324', '202325', '202326', '202327', '202328', '202329', '202330', '202331', '202352']


In [47]:
if sales_check is not None:
    # Kiểm tra riêng week 202153 vì đây là giá trị đã thấy ở Jan-2022.
    target_week = "202153"

    print(
        "202153 exists in Sales:",
        target_week in set(sales_week_key.dropna())
    )

    print(
        "202153 exists in Calendar:",
        target_week in set(calendar_week_key.dropna())
    )

    calendar_202153 = dim_date[
        calendar_week_key == target_week
    ]

    display(calendar_202153)

202153 exists in Sales: True
202153 exists in Calendar: False


,year,month_number,month_name_short,week_number,year_week,week_start_date,week_end_date,cny_flag


In [48]:
if sales_check is not None:
    # Tạo cột chuẩn hóa để dùng ở bước modeling sau này.
    sales_check["week_key"] = normalize_integer_like_key(
        sales_check["week"]
    )

    dim_date["year_week_key"] = normalize_integer_like_key(
        dim_date["year_week"]
    )

    print(
        "Duplicate normalized year_week_key in DimDate:",
        dim_date.duplicated(
            subset=["year_week_key"]
        ).sum()
    )

Duplicate normalized year_week_key in DimDate: 0


## 23.3 Classification grain

`product_style_color` không unique nên không thể merge trực tiếp vào `DimProduct`.

Ta cần xác định **grain thực tế** của bảng Classification bằng cách kiểm tra các tổ hợp key khả dĩ.

In [49]:
print("Classification rows:", len(classification_clean))
print(
    "Exact duplicate full rows:",
    classification_clean.duplicated().sum()
)

grain_candidate_cols = [
    c for c in [
        "product_style_color",
        "sales_season",
        "sales_season_num",
        "launch_season",
        "launch_season_num",
        "final_status",
        "b2c_assortment",
        "b2b_assortment",
        "total_assortment"
    ]
    if c in classification_clean.columns
]

print("Available candidate columns:")
print(grain_candidate_cols)

Classification rows: 150209
Exact duplicate full rows: 1965
Available candidate columns:
['product_style_color', 'sales_season', 'sales_season_num', 'launch_season', 'launch_season_num', 'final_status', 'b2c_assortment', 'b2b_assortment', 'total_assortment']


In [50]:
candidate_keys = [
    ["product_style_color"],
    ["product_style_color", "sales_season"],
    ["product_style_color", "sales_season_num"],
    ["product_style_color", "launch_season"],
    ["product_style_color", "launch_season_num"],
    ["product_style_color", "sales_season", "launch_season"],
    ["product_style_color", "sales_season", "final_status"],
    ["product_style_color", "sales_season", "launch_season", "final_status"],
]

grain_results = []

for key_cols in candidate_keys:
    if all(col in classification_clean.columns for col in key_cols):
        unique_combinations = (
            classification_clean[key_cols]
            .drop_duplicates()
            .shape[0]
        )

        duplicate_rows = classification_clean.duplicated(
            subset=key_cols
        ).sum()

        grain_results.append({
            "candidate_key": " + ".join(key_cols),
            "columns": len(key_cols),
            "unique_combinations": unique_combinations,
            "duplicate_rows": duplicate_rows,
            "uniqueness_pct": round(
                unique_combinations / len(classification_clean) * 100,
                4
            )
        })

grain_check = pd.DataFrame(grain_results).sort_values(
    ["duplicate_rows", "columns"]
)

grain_check

,candidate_key,columns,unique_combinations,duplicate_rows,uniqueness_pct
7,product_style_color + sales_season + launch_se...,4,148215,1994,98.6725
6,product_style_color + sales_season + final_status,3,148214,1995,98.6719
5,product_style_color + sales_season + launch_se...,3,148143,2066,98.6246
1,product_style_color + sales_season,2,148141,2068,98.6233
2,product_style_color + sales_season_num,2,148141,2068,98.6233
3,product_style_color + launch_season,2,12825,137384,8.5381
4,product_style_color + launch_season_num,2,12218,137991,8.1340
0,product_style_color,1,12192,138017,8.1167


In [51]:
# Số record Classification trên mỗi product_style_color.
classification_group_size = (
    classification_clean
    .groupby("product_style_color", dropna=False)
    .size()
    .sort_values(ascending=False)
)

classification_group_size.describe()

count    12192.000000
mean        12.320292
std          5.607747
min          1.000000
25%          9.000000
50%         13.000000
75%         18.000000
max         33.000000
dtype: float64

In [52]:
classification_group_size.head(20)

product_style_color
NaN                                    33
a6ad01de29e34ba7b5e1bcabbffb88deDEN    30
abc9abfa0fcc47708b2523f6665818faDOO    30
3b9739993fa4472aac3d24209270ff0eDOO    30
9be57e2f5162429eb84992a01c4b425eDEN    30
e12f92c571624be79e50c7ee790b5fc1DEN    30
0ec164fada4841998dc3a2fd213e67a0HOG    30
f32997078e7b4f69a803fb6a22452ee5NAU    30
6d142b09d1b24ddeb69a338fada07527DEN    30
cffde9ef555a4e15a1ca868a3708e53bTIM    30
50f1873082264733beb3b53c30d9a83aDEN    30
2b2adde97ca94a3f860339a96080abafXAM    30
30cb17d622ff4887b5907d991e26aa63DOO    30
7bad61e1127e485986ae77e4de482608DEN    30
cd70307276a74ac8b56ba138a626845bXNH    30
7a3e0d721969483aa8c69a436d275b05DOO    30
be3b2dcdaf284abd88c82233b11dcce9DEN    30
1a9ddf9ba8964af3830d0c1f985b02e6DEN    30
02c6178d8950479f9fbac822963b965eDEN    30
2ac39c9307c044acb34719d20eb1dd96DEN    29
dtype: int64

In [53]:
# Xem một product_style_color có nhiều record để hiểu vì sao nó lặp.
sample_style_color = classification_group_size.index[0]

print("Sample product_style_color:", sample_style_color)

classification_clean[
    classification_clean["product_style_color"] == sample_style_color
].sort_values(
    [
        c for c in [
            "sales_season_num",
            "launch_season_num",
            "final_status"
        ]
        if c in classification_clean.columns
    ]
).head(50)

Sample product_style_color: nan


,launch_season,launch_season_num,sales_season,sales_season_num,final_status,b2c_assortment,b2b_assortment,total_assortment,product_style_color


In [54]:
# Kiểm tra merge nguy cơ gây row explosion, KHÔNG dùng kết quả merge này làm DimProduct.
classification_per_style = (
    classification_clean
    .groupby("product_style_color", dropna=False)
    .size()
    .rename("classification_rows")
    .reset_index()
)

product_class_risk = dim_product[
    ["product_id", "product_style_color"]
].merge(
    classification_per_style,
    on="product_style_color",
    how="left"
)

print(
    "Products linked to >1 Classification row:",
    (product_class_risk["classification_rows"].fillna(0) > 1).sum()
)

product_class_risk[
    product_class_risk["classification_rows"].fillna(0) > 1
].sort_values(
    "classification_rows",
    ascending=False
).head(20)

Products linked to >1 Classification row: 11705


,product_id,product_style_color,classification_rows
84598,2b2adde97ca94a3f860339a96080abafXAM43,2b2adde97ca94a3f860339a96080abafXAM,30.0
84593,3b9739993fa4472aac3d24209270ff0eDOO43,3b9739993fa4472aac3d24209270ff0eDOO,30.0
84544,6d142b09d1b24ddeb69a338fada07527DEN43,6d142b09d1b24ddeb69a338fada07527DEN,30.0
84671,e12f92c571624be79e50c7ee790b5fc1DEN43,e12f92c571624be79e50c7ee790b5fc1DEN,30.0
84631,9be57e2f5162429eb84992a01c4b425eDEN43,9be57e2f5162429eb84992a01c4b425eDEN,30.0
86670,0ec164fada4841998dc3a2fd213e67a0HOG39,0ec164fada4841998dc3a2fd213e67a0HOG,30.0
86816,50f1873082264733beb3b53c30d9a83aDEN39,50f1873082264733beb3b53c30d9a83aDEN,30.0
86901,cffde9ef555a4e15a1ca868a3708e53bTIM39,cffde9ef555a4e15a1ca868a3708e53bTIM,30.0
86891,7a3e0d721969483aa8c69a436d275b05DOO39,7a3e0d721969483aa8c69a436d275b05DOO,30.0
86881,f32997078e7b4f69a803fb6a22452ee5NAU39,f32997078e7b4f69a803fb6a22452ee5NAU,30.0


## 23.4 COGS / Retail Price date validity

Hai bảng dùng khoảng hiệu lực `valid_from` → `valid_to`.

Dữ liệu nguồn có dạng ngày kiểu `DD.MM.YYYY` và có thể có sentinel như `01.12.9999`.
Pandas `datetime64[ns]` không hỗ trợ đầy đủ năm 9999, nên phần audit này dùng `datetime.datetime` của Python để kiểm tra mà không làm mất sentinel.

In [55]:
from datetime import datetime, date

# Đọc lại RAW để audit date, tránh dùng cột đã parse sai ở bước trước.
cogs_raw_audit = pd.read_excel(MASTER_PATH / "COGS.xlsx")
retail_raw_audit = pd.read_excel(MASTER_PATH / "Retail_price.xlsx")

print("COGS raw date samples:")
display(cogs_raw_audit[["valid_from", "valid_to"]].head(10))

print("Retail Price raw date samples:")
display(retail_raw_audit[["valid_from", "valid_to"]].head(10))

COGS raw date samples:


,valid_from,valid_to
0,23.12.2021,31.01.2023
1,23.12.2021,31.01.2023
2,01.01.2023,07.02.2023
3,01.01.2023,07.02.2023
4,01.01.2023,07.02.2023
5,01.01.2023,07.02.2023
6,01.01.2023,07.02.2023
7,01.01.2023,07.02.2023
8,01.01.2023,07.02.2023
9,01.01.2023,07.02.2023


Retail Price raw date samples:


,valid_from,valid_to
0,01.01.2024,01.12.9999
1,01.01.2024,01.12.9999
2,01.01.2024,01.12.9999
3,01.01.2024,01.12.9999
4,01.01.2024,01.12.9999
5,01.01.2024,01.12.9999
6,01.01.2024,01.12.9999
7,01.01.2024,01.12.9999
8,01.01.2024,01.12.9999
9,01.01.2024,01.12.9999


In [56]:
def parse_business_date(value):
    """
    Parse an toàn các date thường gặp.
    Trả về Python datetime/date để hỗ trợ year=9999.
    Trả về None nếu không parse được.
    """
    if pd.isna(value):
        return None

    # Excel/pandas datetime
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    if isinstance(value, datetime):
        return value

    if isinstance(value, date):
        return datetime(value.year, value.month, value.day)

    text = str(value).strip()

    formats = [
        "%d.%m.%Y",
        "%d/%m/%Y",
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass

    return None


def audit_date_range(df, table_name):
    audit = df.copy()

    audit["valid_from_parsed"] = audit["valid_from"].apply(
        parse_business_date
    )
    audit["valid_to_parsed"] = audit["valid_to"].apply(
        parse_business_date
    )

    audit["valid_from_parse_failed"] = (
        audit["valid_from"].notna()
        & audit["valid_from_parsed"].isna()
    )

    audit["valid_to_parse_failed"] = (
        audit["valid_to"].notna()
        & audit["valid_to_parsed"].isna()
    )

    audit["invalid_range"] = audit.apply(
        lambda row: (
            row["valid_from_parsed"] is not None
            and row["valid_to_parsed"] is not None
            and row["valid_from_parsed"] > row["valid_to_parsed"]
        ),
        axis=1
    )

    audit["open_ended_9999"] = audit["valid_to_parsed"].apply(
        lambda x: x is not None and x.year == 9999
    )

    summary = pd.Series({
        "table": table_name,
        "rows": len(audit),
        "missing_valid_from": audit["valid_from"].isna().sum(),
        "missing_valid_to": audit["valid_to"].isna().sum(),
        "valid_from_parse_failed":
            audit["valid_from_parse_failed"].sum(),
        "valid_to_parse_failed":
            audit["valid_to_parse_failed"].sum(),
        "invalid_from_gt_to":
            audit["invalid_range"].sum(),
        "open_ended_year_9999":
            audit["open_ended_9999"].sum()
    })

    return audit, summary

In [57]:
cogs_date_audit, cogs_date_summary = audit_date_range(
    cogs_raw_audit,
    "COGS"
)

retail_date_audit, retail_date_summary = audit_date_range(
    retail_raw_audit,
    "Retail Price"
)

date_validity_summary = pd.DataFrame([
    cogs_date_summary,
    retail_date_summary
])

date_validity_summary

,table,rows,missing_valid_from,missing_valid_to,valid_from_parse_failed,valid_to_parse_failed,invalid_from_gt_to,open_ended_year_9999
0,COGS,522675,0,0,0,0,0,433433
1,Retail Price,669460,0,0,0,0,0,212350


In [58]:
print("COGS parse failures:")
display(
    cogs_date_audit[
        cogs_date_audit["valid_from_parse_failed"]
        | cogs_date_audit["valid_to_parse_failed"]
    ][
        [
            "product_id",
            "valid_from",
            "valid_to"
        ]
    ].head(20)
)

print("Retail Price parse failures:")
display(
    retail_date_audit[
        retail_date_audit["valid_from_parse_failed"]
        | retail_date_audit["valid_to_parse_failed"]
    ][
        [
            "product_id",
            "valid_from",
            "valid_to"
        ]
    ].head(20)
)

COGS parse failures:


,product_id,valid_from,valid_to


Retail Price parse failures:


,product_id,valid_from,valid_to


In [59]:
print("COGS invalid date ranges:")
display(
    cogs_date_audit[
        cogs_date_audit["invalid_range"]
    ][
        [
            "product_id",
            "valid_from",
            "valid_to",
            "valid_from_parsed",
            "valid_to_parsed"
        ]
    ].head(20)
)

print("Retail Price invalid date ranges:")
display(
    retail_date_audit[
        retail_date_audit["invalid_range"]
    ][
        [
            "product_id",
            "valid_from",
            "valid_to",
            "valid_from_parsed",
            "valid_to_parsed"
        ]
    ].head(20)
)

COGS invalid date ranges:


,product_id,valid_from,valid_to,valid_from_parsed,valid_to_parsed


Retail Price invalid date ranges:


,product_id,valid_from,valid_to,valid_from_parsed,valid_to_parsed


In [60]:
print("COGS rows with year 9999 as open-ended valid_to:")
display(
    cogs_date_audit[
        cogs_date_audit["open_ended_9999"]
    ][
        [
            "product_id",
            "amount",
            "valid_from",
            "valid_to"
        ]
    ].head(10)
)

print("Retail Price rows with year 9999 as open-ended valid_to:")
display(
    retail_date_audit[
        retail_date_audit["open_ended_9999"]
    ][
        [
            "product_id",
            "amount",
            "valid_from",
            "valid_to"
        ]
    ].head(10)
)

COGS rows with year 9999 as open-ended valid_to:


,product_id,amount,valid_from,valid_to
89242,f64e53d83cfa461abe156559d55ccae2DEN35,85000,09.02.2023,01.12.9999
89243,c50a6b0caa0b4ca9af55cf27607b64d3DEN36,85000,09.02.2023,01.12.9999
89244,408592eb75e44216a1f65e831f9a08e5DEN37,85000,09.02.2023,01.12.9999
89245,9e7e6f15b9764540bebaca2de63abb03DEN38,85000,09.02.2023,01.12.9999
89246,473397f29c1a4005b3e9bf75496f496eDEN39,85000,09.02.2023,01.12.9999
89247,5d8777c2f52545f99aa1385061d1c1e0DEN40,85000,09.02.2023,01.12.9999
89248,34eead30886c43fba0f8e2ec0021805eDEN41,85000,09.02.2023,01.12.9999
89249,eb26e54d6d044983b47f873c48f79036DEN42,85000,09.02.2023,01.12.9999
89250,46af99513bbb44159db6fbe4ab5c0569DEN43,85000,09.02.2023,01.12.9999
89251,2fe721fe167f405d9df1d6868f22d820DEN44,85000,09.02.2023,01.12.9999


Retail Price rows with year 9999 as open-ended valid_to:


,product_id,amount,valid_from,valid_to
0,f64e53d83cfa461abe156559d55ccae2DEN35,125000,01.01.2024,01.12.9999
1,c50a6b0caa0b4ca9af55cf27607b64d3DEN36,125000,01.01.2024,01.12.9999
2,408592eb75e44216a1f65e831f9a08e5DEN37,125000,01.01.2024,01.12.9999
3,9e7e6f15b9764540bebaca2de63abb03DEN38,125000,01.01.2024,01.12.9999
4,473397f29c1a4005b3e9bf75496f496eDEN39,125000,01.01.2024,01.12.9999
5,5d8777c2f52545f99aa1385061d1c1e0DEN40,125000,01.01.2024,01.12.9999
6,34eead30886c43fba0f8e2ec0021805eDEN41,125000,01.01.2024,01.12.9999
7,eb26e54d6d044983b47f873c48f79036DEN42,125000,01.01.2024,01.12.9999
8,46af99513bbb44159db6fbe4ab5c0569DEN43,125000,01.01.2024,01.12.9999
9,2fe721fe167f405d9df1d6868f22d820DEN44,125000,01.01.2024,01.12.9999


### Cách đọc kết quả COGS / Retail Price

- `parse_failed = 0` là lý tưởng.
- `valid_from > valid_to` phải được điều tra vì khoảng hiệu lực không hợp lệ.
- `valid_to` năm 9999 thường là sentinel cho "còn hiệu lực", không nên parse bằng `pd.to_datetime` rồi làm mất hoặc đảo ngày.
- Vì COGS/Retail Price thay đổi theo thời gian, grain gần với `product_id + effective date range`, không phải 1 row/product.

## 24. Relationship Readiness Checkpoint

Sau khi chạy 4 phần trên, **chưa cần sửa/xóa dữ liệu ngay**. Hãy dùng output để quyết định:

- khóa phù hợp cho Sales ↔ Distribution;
- cách chuẩn hóa Calendar key;
- grain thật của Classification;
- cách xử lý effective date của COGS/Retail Price.

Khi các quan hệ này rõ ràng, mới thiết kế Star Schema để tránh many-to-many và double counting.

In [61]:
print("CHECKPOINT")
print("1. Review Sales ↔ Distribution customer/pair coverage")
print("2. Review Calendar coverage after key normalization")
print("3. Review Classification candidate grain table")
print("4. Review COGS/Retail date validity summary")

CHECKPOINT
1. Review Sales ↔ Distribution customer/pair coverage
2. Review Calendar coverage after key normalization
3. Review Classification candidate grain table
4. Review COGS/Retail date validity summary
